# Telecom Encaissement — Cash vs Online forecast
Simple steps: load → clean → daily aggregate → plot → train many models → compare → save best.

In [ ]:
# Library imports: pandas = tables, numpy = numbers, matplotlib = plots
import pandas as pd  # dataframes for loading/cleaning/aggregating
import numpy as np  # numeric helpers (arange, where)
import matplotlib.pyplot as plt  # line graphs over time

In [ ]:
# Path of the raw Excel file in this folder
DATA = "encaissement du 01-01-2026.xlsx"
# Open it once (8 monthly sheets: janvier..aout) so later reads are fast
xls = pd.ExcelFile(DATA)
# Show sheet names to confirm we have all 8 months
print("Sheets:", xls.sheet_names)

In [ ]:
# Peek at 3 rows per sheet: check shape and column names
# (headers differ slightly per month, e.g. 'Date de tr' vs 'Date de transaction')
for s in xls.sheet_names:  # loop over the 8 month sheets
    d = pd.read_excel(xls, sheet_name=s, nrows=3)  # read only 3 rows, fast preview
    print(s, d.shape, list(d.columns))  # show month, shape, headers

In [ ]:
# Full load: read every month sheet, standardize by column POSITION (headers vary)
frames = []  # one dataframe per month, merged below
for sheet in xls.sheet_names:  # janvier, fevrier, ... aout
    df = pd.read_excel(xls, sheet_name=sheet)  # full month data
    df = df.iloc[:, :12]  # keep the first 12 columns only
    # Rename to one standard schema (position-based, same for all months)
    df.columns = ["method", "cash_raw", "timbre", "total", "cheque", "virement",
                  "autres", "no_cheque", "code_banque", "dir_banque", "date", "dept"]
    frames.append(df)  # collect this month
raw = pd.concat(frames, ignore_index=True)  # stack all months into one table (~627k rows)
print("raw rows:", len(raw))  # confirm total row count

In [ ]:
# Inspect raw payment-method labels before cleaning
# Expect 'Especes' + online typos ('paiement en ligne', 'paiment...', 'paiemnt...') + 1 total row
print(raw["method"].astype(str).value_counts(dropna=False).head(10))

In [ ]:
# Normalize method to exactly 'cash' or 'online'
df = raw.copy()  # work on a copy, keep raw untouched
m = df["method"].astype(str).str.lower().str.strip()  # lowercase+trim for matching
is_online = m.str.contains("ligne|paiment|paiemnt", na=False)  # online + all typos
is_cash = m.str.contains("esp", na=False)  # 'Especes' (cash)
df["pay"] = "other"  # default label, removed below (catches the total row)
df.loc[is_online, "pay"] = "online"  # tag online rows
df.loc[is_cash, "pay"] = "cash"  # tag cash rows
df = df[df["pay"].isin(["cash", "online"])]  # keep only the two real methods
print(df["pay"].value_counts())  # check balance: cash dominates

In [ ]:
# Parse dates to day precision; aout holds datetime strings with time, others are dates
df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True).dt.normalize()
df = df.dropna(subset=["date"])  # drop rows with unparseable dates
print("date range:", df["date"].min(), "->", df["date"].max())  # expect 2026-01-01 -> 2026-08-31

In [ ]:
# Build row-level amounts. Key fact: cash money is in `total`, online money in `autres`
for c in ["total", "autres"]:  # ensure numeric (text/empty -> 0)
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["cash_amount"] = 0.0  # init: 0 unless this row is cash
df["online_amount"] = 0.0  # init: 0 unless this row is online
# Cash rows take their value from `total` (online rows stay 0 here)
df.loc[df["pay"] == "cash", "cash_amount"] = df.loc[df["pay"] == "cash", "total"]
# Online rows take their value from `autres` (their `total` is always 0)
df.loc[df["pay"] == "online", "online_amount"] = df.loc[df["pay"] == "online", "autres"]
print("cash sum:", df["cash_amount"].sum())  # total cash value
print("online sum:", df["online_amount"].sum())  # total online value

In [ ]:
# Sanity check: group by (date, method) to preview daily sums and counts
g = df.groupby([df["date"], df["pay"]]).agg(  # one row per day+method
    amount=("cash_amount", "sum"),  # daily cash sum
    online_amt=("online_amount", "sum"),  # daily online sum
    n=("pay", "size"))  # daily transaction count
print(g.head(6).to_string())  # show first days

In [ ]:
# Aggregate to ONE row per calendar day (the time series we forecast)
# Sum amounts per method, count txns per method, then reindex on full calendar
pv_amount_cash = df[df["pay"]=="cash"].pivot_table(index="date", values="cash_amount", aggfunc="sum")
pv_amount_online = df[df["pay"]=="online"].pivot_table(index="date", values="online_amount", aggfunc="sum")
pv_count = df.pivot_table(index="date", columns="pay", aggfunc="size", fill_value=0)
# Full calendar 2026-01-01..2026-08-31 so missing days (e.g. 2026-01-02) become 0
daily = pd.DataFrame(index=pd.date_range(df["date"].min(), df["date"].max(), freq="D"))
daily.index.name = "date"  # name the index for reset_index below
daily["cash_amount"] = pv_amount_cash.reindex(daily.index, fill_value=0)  # daily cash sums
daily["online_amount"] = pv_amount_online.reindex(daily.index, fill_value=0)  # daily online sums
daily["cash_count"] = pv_count["cash"].reindex(daily.index, fill_value=0) if "cash" in pv_count else 0
daily["online_count"] = pv_count["online"].reindex(daily.index, fill_value=0) if "online" in pv_count else 0
daily = daily.reset_index()  # date back to a column (243 rows x 5 cols)
print(daily.shape)  # expect (243, 5)
print(daily.head().to_string())  # show first days

In [ ]:
# Summary stats of the daily series: mean/min/max per target (spot outliers, scale)
print(daily.describe().round(1).to_string())

In [ ]:
# Line plot: daily amounts over time, cash vs online
plt.figure(figsize=(12, 4))  # wide figure for 243 days
plt.plot(daily["date"], daily["cash_amount"], label="cash")  # cash curve
plt.plot(daily["date"], daily["online_amount"], label="online")  # online curve
plt.title("Daily amounts: cash vs online")  # chart title
plt.legend()  # show cash/online labels
plt.show()  # display the figure

In [ ]:
# Line plot: daily transaction COUNTS over time, cash vs online
plt.figure(figsize=(12, 4))  # same wide format
plt.plot(daily["date"], daily["cash_count"], label="cash")  # cash txn count
plt.plot(daily["date"], daily["online_count"], label="online")  # online txn count
plt.title("Daily counts: cash vs online")  # chart title
plt.legend()  # show labels
plt.show()  # display the figure

## Insights
- Cash dominates value and volume.
- Online grows from June onward (June 1593, July 3280, Aug 3156 txns).
- Median ticket ~1200 DZD for both methods.
- We forecast 4 daily targets: cash_amount, online_amount, cash_count, online_count.

In [ ]:
# Monthly totals: confirm online growth trend (June onward)
tmp = daily.copy()  # copy to add a month column safely
tmp["month"] = tmp["date"].dt.strftime("%Y-%m")  # 'YYYY-MM' label per day
# Sum the 4 targets per month and print
print(tmp.groupby("month")[["cash_amount", "online_amount", "cash_count", "online_count"]].sum().to_string())

In [ ]:
# Define the 4 forecast targets and the 8 time features every model uses
TARGETS = ["cash_amount", "online_amount", "cash_count", "online_count"]  # what we predict
FEATURES = ["trend", "dow", "month", "dom", "weekend", "lag_1", "lag_7", "roll_7"]  # inputs
print(TARGETS)  # show targets
print(FEATURES)  # show features

In [ ]:
# Feature builder for ONE target: calendar features + recent history (lags)
def add_features(daily_df, target):  # daily_df = daily table, target = one of TARGETS
    d = daily_df[["date", target]].copy()  # keep only date + target column
    d["trend"] = np.arange(len(d))  # days since start (trend)
    d["dow"] = d["date"].dt.dayofweek  # 0=Mon..6=Sun (weekly pattern)
    d["month"] = d["date"].dt.month  # 1..12 (seasonality)
    d["dom"] = d["date"].dt.day  # day of month 1..31
    d["weekend"] = (d["dow"] >= 5).astype(int)  # 1 if Sat/Sun else 0
    d["lag_1"] = d[target].shift(1)  # yesterday's value
    d["lag_7"] = d[target].shift(7)  # same weekday last week
    d["roll_7"] = d[target].shift(1).rolling(7).mean()  # mean of last 7 days
    return d.dropna().reset_index(drop=True)  # drop first 7 rows (lags missing)

In [ ]:
# Candidate regressors compared for EVERY target (sklearn only, kept simple)
from sklearn.linear_model import LinearRegression, Ridge  # linear baselines
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor  # tree models
MODELS = {  # 4 models to compare per target
    "LinearRegression": LinearRegression(),  # plain linear
    "Ridge": Ridge(),  # linear with L2 regularization
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),  # 200 trees
    "GradientBoosting": GradientBoostingRegressor(random_state=42),  # boosted trees
}
print(list(MODELS))  # show model names

In [ ]:
# Train + compare: per target, chronological 80/20 split, score MAE/RMSE/R2
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # metrics
results = []  # one dict per (target, model)
for target in TARGETS:  # 4 targets
    feat = add_features(daily, target)  # build features for this target
    n = len(feat)  # usable rows (~236 after lags)
    split = int(n * 0.8)  # 80% train (past), 20% test (recent) - time order kept
    X = feat[FEATURES]  # input matrix
    y = feat[target]  # output vector
    Xtr, Xte = X.iloc[:split], X.iloc[split:]  # time split of inputs
    ytr, yte = y.iloc[:split], y.iloc[split:]  # time split of outputs
    for name, model in MODELS.items():  # try each of the 4 models
        model.fit(Xtr, ytr)  # train on past
        p = model.predict(Xte)  # predict recent (held-out)
        results.append({"target": target, "model": name,  # store scores
            "MAE": mean_absolute_error(yte, p),  # mean abs error (main criterion)
            "RMSE": mean_squared_error(yte, p) ** 0.5,  # root mean sq error
            "R2": r2_score(yte, p)})  # variance explained
print(pd.DataFrame(results).round(2).to_string())  # 16-row comparison table

In [ ]:
# Pick winner per target = lowest MAE on the held-out test split
best_names = {}  # e.g. {'cash_amount': 'GradientBoosting', ...}
for target in TARGETS:  # per target
    comp = [r for r in results if r["target"] == target]  # its 4 model rows
    winner = sorted(comp, key=lambda r: r["MAE"])[0]["model"]  # smallest MAE wins
    best_names[target] = winner  # record winner name
print(best_names)  # show the 4 winners

In [ ]:
# Refit each winner on ALL days (train+test) for the final served model
best_models = {}  # target -> fitted model object
for target in TARGETS:  # per target
    feat = add_features(daily, target)  # full features (all ~236 rows)
    X = feat[FEATURES]  # full inputs
    y = feat[target]  # full outputs
    # Fresh instance of the winning model type (same hyperparams as compared)
    m = {"LinearRegression": LinearRegression(), "Ridge": Ridge(),
         "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
         "GradientBoosting": GradientBoostingRegressor(random_state=42)}[best_names[target]]
    m.fit(X, y)  # fit on everything - this is what gets saved/served
    best_models[target] = m  # store fitted model
print("refit done")  # confirm

In [ ]:
# Recursive multi-day forecast: predict t+1, feed it back as lag for t+2, ...
def forecast_target(target, days=30):  # target = one of TARGETS, days = horizon
    hist = daily.set_index("date")[target].copy()  # history series to extend
    last = hist.index.max()  # last real date (2026-08-31)
    dates, preds = [], []  # outputs
    for i in range(1, days + 1):  # step day by day into the future
        d = last + pd.Timedelta(days=i)  # future date
        lag_1 = hist.iloc[-1]  # yesterday (real or already-predicted)
        lag_7 = hist.iloc[-7] if len(hist) >= 7 else hist.iloc[0]  # same weekday
        roll_7 = hist.iloc[-7:].mean() if len(hist) >= 7 else hist.mean()  # 7-day mean
        # One-row feature frame in the same FEATURES order as training
        row = pd.DataFrame([{"trend": len(daily) - 1 + i, "dow": d.dayofweek,
            "month": d.month, "dom": d.day, "weekend": int(d.dayofweek >= 5),
            "lag_1": lag_1, "lag_7": lag_7, "roll_7": roll_7}])[FEATURES]
        v = float(max(0, best_models[target].predict(row)[0]))  # predict, clip negatives to 0
        dates.append(d)  # collect date
        preds.append(v)  # collect value
        hist.loc[d] = v  # feed prediction back as history for next step
    return dates, preds  # future dates + values

In [ ]:
# Demo: forecast 30 days for each target, print first 3 values as a smoke test
for t in TARGETS:  # 4 targets
    _, p = forecast_target(t, 30)  # 30-day forecast (dates ignored here)
    print(t, [round(x, 1) for x in p[:3]])  # show first 3 predicted values

In [ ]:
# Save everything the app needs into best_model.pkl (no retraining at serve time)
import pickle  # stdlib serializer for python objects
bundle = {"models": best_models, "best_names": best_names,  # fitted models + winner names
    "metrics": results, "feature_cols": FEATURES,  # comparison table + feature order
    "daily": daily, "targets": TARGETS}  # history + target list (for lags/trend)
with open("best_model.pkl", "wb") as f:  # write binary pickle next to the notebook
    pickle.dump(bundle, f)  # serialize the dict
print("saved best_model.pkl:", best_names)  # confirm winners saved